# Cardiovascular Disease (CVD) Prediction

Predicting the presence of cardiovascular disease from patient health metrics using XGBoost.

**Dataset:** `cardio_train.csv` — 70,000 patient records ([Kaggle](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset))  
**Target:** `cardio` (1 = has CVD, 0 = no CVD)  
**Pipeline:** Cleaning → Feature Engineering → Train/Test Split → Scaling → Cross-Validation → Evaluation


## 1. Imports

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

# Project root is one level up from notebooks/
ROOT = Path().resolve().parent
DATA_DIR   = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print('Project root:', ROOT)
print('Data path:   ', DATA_DIR / 'cardio_train.csv')


## 2. Load Data

The CSV uses semicolons as delimiters.  
Path resolves to `../data/cardio_train.csv` relative to this notebook.


In [ ]:
df = pd.read_csv(DATA_DIR / 'cardio_train.csv', sep=';')
print(f'Shape: {df.shape}')
df.head()


## 3. Exploratory Data Analysis

Before cleaning we look at data types, missing values, and class balance.


In [ ]:
df.info()


In [ ]:
print('Age stats (raw, in days):')
print(df['age'].describe())
print()
print('Target balance:')
print(df['cardio'].value_counts())
print(df['cardio'].value_counts(normalize=True).round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
df['cardio'].value_counts().rename({0: 'No CVD', 1: 'CVD'}).plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white'
)
axes[0].set_title('Target Class Distribution')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Age distribution
age_years = (df['age'] / 365.25).round().astype(int)
age_years.plot(kind='hist', ax=axes[1], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Age Distribution (years)')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()


## 4. Data Cleaning

- Convert age from days → years
- Remove duplicate rows (excluding `id`)
- Drop rows with biologically impossible clinical values
- Drop any remaining nulls


In [ ]:
df = df.copy()

# Convert age from days to years
df['age'] = (df['age'] / 365.25).round().astype(int)

# Remove duplicate rows (ignore id column)
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'id'])

# Drop rows with biologically impossible clinical values
df = df[
    df['ap_hi'].between(90, 250)
    & df['ap_lo'].between(60, 180)
    & (df['ap_hi'] > df['ap_lo'])
    & df['height'].between(100, 220)
    & df['weight'].between(30, 200)
    & df['cholesterol'].isin([1, 2, 3])
    & df['gluc'].isin([1, 2, 3])
]

# Drop remaining nulls
df = df.dropna()

print(f'Rows after cleaning: {len(df):,}')


## 5. Feature Engineering

| Feature | Meaning |
|---|---|
| `bmi` | Body Mass Index — weight / height² |
| `obesity` | BMI ≥ 30 |
| `overweight` | 25 ≤ BMI < 30 |
| `hypertension` | Systolic BP ≥ 140 mmHg |
| `pulse_pressure` | ap_hi − ap_lo (arterial stiffness proxy) |
| `age_bmi` | Interaction term: age × BMI / 100 |


In [ ]:
df['bmi']            = df['weight'] / (df['height'] / 100) ** 2
df['obesity']        = (df['bmi'] >= 30).astype(int)
df['overweight']     = ((df['bmi'] >= 25) & (df['bmi'] < 30)).astype(int)
df['hypertension']   = (df['ap_hi'] >= 140).astype(int)
df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']
df['age_bmi']        = df['age'] * df['bmi'] / 100

print('Top correlations with target:')
print(df.corrwith(df['cardio']).abs().sort_values(ascending=False).head(10))


## 6. Prepare Features and Target


In [ ]:
target   = df['cardio']
features = df.drop(columns=['id', 'cardio'])

print('Features:', features.columns.tolist())
print('Shape:   ', features.shape)


## 7. Train / Test Split

> ⚠️ Split happens **before** scaling to prevent data leakage.
> The scaler must only learn statistics from training data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target,
    test_size=0.2,
    random_state=42,
    stratify=target,
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')


## 8. Feature Scaling

`fit_transform` on train only — `transform` on test. Never the other way around.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


## 9. Cross-Validation

Using a `Pipeline` ensures scaling is re-fitted inside each fold —
the validation fold is never seen during `fit_transform`.


In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        verbosity=0,
        eval_metric='logloss',
    ))
])

sss = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
cv_auc = cross_val_score(pipeline, features, target, cv=sss, scoring='roc_auc')

print(f'AUC per fold: {cv_auc.round(4)}')
print(f'Mean AUC:     {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')


## 10. Final Model — Training and Evaluation


In [ ]:
model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    verbosity=0,
    eval_metric='logloss',
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'AUC:      {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No CVD', 'CVD']))


## 11. Visualizations


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No CVD', 'CVD']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion Matrix')

# ROC Curve
auc_score = roc_auc_score(y_test, y_prob)
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], name='XGBoost')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random baseline')
axes[1].set_title(f'ROC Curve  (AUC = {auc_score:.3f})')
axes[1].legend()

# Feature Importance
importances = pd.Series(model.feature_importances_, index=features.columns)
importances.sort_values().plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Feature Importances (XGBoost)')
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()


## 12. Save Model

Both the model and scaler are saved — you need both to predict on new data.


In [ ]:
joblib.dump(model,  MODELS_DIR / 'xgb_cvd_model.pkl')
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')

print('Saved:', MODELS_DIR / 'xgb_cvd_model.pkl')
print('Saved:', MODELS_DIR / 'scaler.pkl')
